<a href="https://colab.research.google.com/github/gokul-pv/JobProfile-Resume-Matching/blob/master/EVA6_Session3_Assignment_Working_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from __future__ import print_function
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from PIL import Image

import numpy as np

In [ ]:
from torch.utils.data import Dataset

class ExampleDataset(Dataset):
    def __init__(self, dataMNIST, transform=None):
        self.x1 = dataMNIST.data
        self.x2 = torch.randint(low=0,high=10,size=(1,len(dataMNIST))).squeeze()
        self.y1 = dataMNIST.targets
        self.y2 = self.y1 + self.x2
        self.transform = transform

    def __len__(self):
        return len(self.x1)
    
    def __getitem__(self, idx):
        (x1, x2, y1, y2) = (self.x1[idx], self.x2[idx], self.y1[idx], self.y2[idx])

        # doing this so that it is consistent with all other datasets
        # to return a PIL Image
        x1 = Image.fromarray(x1.numpy(), mode='L')    

        if self.transform is not None:
            x1 = self.transform(x1)
            
        return (x1, x2, y1, y2)

In [ ]:
 train_img = datasets.MNIST(
            root='../data',
            train=True,
            download=True,
            )
train_set = ExampleDataset(train_img, transform=transforms.Compose
                           ([transforms.ToTensor(),transforms.Normalize((0.1307,), (0.3081,))]))

In [ ]:
test_img = datasets.MNIST('../data', train=False)

test_set = ExampleDataset(test_img, transform=transforms.Compose
                          ([transforms.ToTensor(),transforms.Normalize((0.1307,), (0.3081,))]))

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1) #input -? OUtput? RF
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv4 = nn.Conv2d(128, 256, 3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.conv5 = nn.Conv2d(256, 512, 3)
        self.conv6 = nn.Conv2d(512, 1024, 3)
        self.conv7 = nn.Conv2d(1024, 10, 3)

        self.fc1 = nn.Linear(10+10, 60)
        self.fc2 = nn.Linear(60, 60)
        self.out = nn.Linear(60, 19)

    def forward(self, x1, x2):
        x1 = self.pool1(F.relu(self.conv2(F.relu(self.conv1(x1)))))
        x1 = self.pool2(F.relu(self.conv4(F.relu(self.conv3(x1)))))
        x1 = F.relu(self.conv6(F.relu(self.conv5(x1))))
        x1 = (self.conv7(x1))
        x1 = x1.view(-1, 10)
        y1 = F.log_softmax(x1)

        y1_temp = y1.argmax(dim=1, keepdim=True)
        
        y1_temp = F.one_hot(y1_temp.squeeze(), num_classes=10)
        x2      = F.one_hot(x2, num_classes=10)

        y1_temp = torch.cat((y1_temp, x2),dim = 1).float()


        y2 = F.relu(self.fc1(y1_temp))
        y2 = F.relu(self.fc2(y2))
        y2 = self.out(y2)

        return y1, F.log_softmax(y2)

In [ ]:
!pip install torchsummary
from torchsummary import summary
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
model = Net().to(device)
# summary(model, input_size=(1, 28, 28))

In [ ]:
torch.manual_seed(1)
batch_size = 128

kwargs = {'num_workers': 1, 'pin_memory': True} if use_cuda else {}

train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True, **kwargs)

test_loader = torch.utils.data.DataLoader(test_set,batch_size=batch_size, shuffle=True, **kwargs)

In [ ]:
from tqdm import tqdm

def train(model, device, train_loader, optimizer, epoch):
    model.train()
    pbar = tqdm(train_loader)
    for batch_idx, (data, x2, target, y2) in enumerate(pbar):
        data, x2, target, y2 = data.to(device), x2.to(device), target.to(device), y2.to(device)
        
        optimizer.zero_grad()
        output1 , output2 = model(data, x2)

        loss1 = F.nll_loss(output1, target)
        loss2 = F.nll_loss(output2, y2)
        
        loss = loss1 + loss2
        loss.backward()
        optimizer.step()

        pbar.set_description(desc= f'loss={loss.item()} batch_id={batch_idx}')   



def test(model, device, test_loader):
    model.eval()
    test_loss1 = 0
    test_loss2 = 0
    correct1 = 0
    correct2 = 0
    with torch.no_grad():
        for data, x2, target, y2 in test_loader:
            data, x2, target, y2 = data.to(device), x2.to(device), target.to(device), y2.to(device)

            output1 , output2 = model(data, x2)
           
            test_loss1 += F.nll_loss(output1, target, reduction='sum').item()  # sum up batch loss
            test_loss2 += F.nll_loss(output2, y2, reduction='sum').item()  # sum up batch loss

            pred1 = output1.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
            pred2 = output2.argmax(dim=1, keepdim=True)


            correct1 += pred1.eq(target.view_as(pred1)).sum().item()
            correct2 += pred2.eq(y2.view_as(pred2)).sum().item()


    test_loss1 /= len(test_loader.dataset)
    test_loss2 /= len(test_loader.dataset)

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss1, correct1, len(test_loader.dataset),
        100. * correct1 / len(test_loader.dataset)))
    
    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss2, correct2, len(test_loader.dataset),
        100. * correct2 / len(test_loader.dataset)))   

In [ ]:
model = Net().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

#optimizer = optim.AdamW(model.parameters(),lr=0.1)

In [ ]:
for epoch in range(1, 2):
    train(model, device, train_loader, optimizer, epoch)
    # test(model, device, test_loader)

In [ ]:
test(model, device, test_loader)